# Linear SVM

In [1]:
# Basic Imports
import sys
import pandas as pd
import time
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Importing shared functions
from src.preprocessing import preprocess
from src.vectorization import vectorize
from src.evaluation import evaluate
from src.experiments import run_experiment

# Model-specific imports
from sklearn.svm import LinearSVC, SVC

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df_raw = pd.read_csv("../data/Dataset.csv")

## Model 1 

Model - Linear SVM     
Vectorization - TF-IDF with N-grams  
Max iterations - 1000 (5000 for experiments)           

### Baseline:

Features -      
* TF-IDF 
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time : 44.5s   
* Training Time : 0.9392 s              

Metrics -           
* Accuracy:  0.97841
* Precision:  0.97294
* Recall:  0.98511
* F1:  0.97839
* Confusion Matrix:         
  [11316    333         -> False Positives             
   181     11975]      -> False Negatives              

In [ ]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df_raw, check=True)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, method = "tfidf")

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"\nFeatures created in {creation_time:.4f} seconds.")
print("Total features = ",total_features)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1

Features created in 43.6435 seconds.
Total features =  477076


In [4]:
# Training model
start_time = time.perf_counter()

model = LinearSVC(random_state=42).fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 0.9126 seconds.


In [5]:
# Evaluation
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9784078975005251
Precision:  0.9729444263893403
Recall:  0.9851102336294834
F1:  0.9783913374237649

Confusion Matrix:
 [[11316   333]
 [  181 11975]]

Top 5 Negative Features:
                   Feature  Importance
352973             reuters  -11.698385
361547                said   -6.373156
1810             breitbart   -6.180282
432972              trumps   -3.986831
454520  washington reuters   -3.545097

Top 5 Positive Features:
          Feature  Importance
446457        via    5.260668
15125       video    3.616887
205873      image    3.371605
205953  image via    2.939298
1777     breaking    2.774254


In [6]:
# Storing results
results = []

results.append({
    "Experiment": "Baseline",
    "Vectorizer": "tfidf",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Features": total_features,
    "Feature Time (s)": creation_time,
    "Training Time (s)": training_time
})

## Experiments

In [7]:
model = LinearSVC(random_state=42, max_iter=5000)

### Experiment 1 (Varying regularization parameter C)

In [8]:
C_var = [0.01, 0.1, 0.5, 2, 10] # Default = 1.0
for var in C_var:
    model_var = LinearSVC(random_state=42,
                       max_iter=5000,
                       C=var)
    results.append(
    run_experiment(
        df_raw,
        model=model_var,
        vectorizer="tfidf",
        experiment_name=f"C={var}",
        ngramRange=(1,2)
    )
)


Features created in 43.9596 seconds.
Total features =  477076
Training completed in 0.5157 seconds.
Accuracy:  0.9477000630119723
Precision:  0.9330793045963325
Recall:  0.9669299111549852
F1:  0.9476169879315897

Confusion Matrix:
 [[10806   843]
 [  402 11754]]

Top 5 Negative Features:
           Feature  Importance
361547        said   -3.685511
352973     reuters   -2.852876
1810     breitbart   -2.846041
432972      trumps   -1.362825
15915   york times   -1.261799

Top 5 Positive Features:
         Feature  Importance
15125      video    2.193724
446457       via    1.083500
1777    breaking    0.917409
197585   hillary    0.885097
205873     image    0.853568

Features created in 44.7007 seconds.
Total features =  477076
Training completed in 0.3977 seconds.
Accuracy:  0.971854652383953
Precision:  0.9641933397995474
Recall:  0.9813260941099046
F1:  0.9718286658931121

Confusion Matrix:
 [[11206   443]
 [  227 11929]]

Top 5 Negative Features:
                   Feature  Impor

### Experiment 2 (Varying min_df)

In [9]:
mdf_var = [2, 10, 20] 
for var in mdf_var:
    results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name=f"mdf={var}",
        ngramRange=(1,2),
        mdf=var
    )
)


Features created in 44.2148 seconds.
Total features =  550000
Training completed in 0.9427 seconds.
Accuracy:  0.9793320730938878
Precision:  0.9744549300357956
Recall:  0.985357025337282
F1:  0.9793169827384718

Confusion Matrix:
 [[11335   314]
 [  178 11978]]

Top 5 Negative Features:
                   Feature  Importance
412950             reuters  -11.430611
3627             breitbart   -6.540351
422179                said   -6.359340
500749              trumps   -3.880843
523666  washington reuters   -3.415056

Top 5 Positive Features:
          Feature  Importance
515119        via    5.141003
45708       video    3.814318
255233      image    3.303492
255316  image via    2.867390
3558     breaking    2.845757

Features created in 43.7462 seconds.
Total features =  195987
Training completed in 0.7078 seconds.
Accuracy:  0.9791220331863054
Precision:  0.9737505079236083
Recall:  0.9856860809476802
F1:  0.979106166382959

Confusion Matrix:
 [[11326   323]
 [  174 11982]]

Top 5

### Experiment 3 (Remove Source Identifiers)

In [10]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Removed source names",
        remove_sourcewords=True
    )
)


Features created in 52.8444 seconds.
Total features =  475888
Training completed in 1.0182 seconds.
Accuracy:  0.971518588531821
Precision:  0.9675737330943458
Recall:  0.976966107272129
F1:  0.9714989420566307

Confusion Matrix:
 [[11251   398]
 [  280 11876]]

Top 5 Negative Features:
                 Feature  Importance
360372              said   -7.342973
15784                000   -4.903089
167629            follow   -4.523350
431789            trumps   -4.217681
321424  president donald   -4.046584

Top 5 Positive Features:
          Feature  Importance
445271        via    6.220171
14943       video    4.325486
205385      image    3.947081
205465  image via    3.467837
287406    october    3.287893


### Experiment 4 (Unigrams Only)

In [11]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Unigrams Only",
        ngramRange=(1,1)
    )
)


Features created in 17.3633 seconds.
Total features =  70042
Training completed in 0.5441 seconds.
Accuracy:  0.9758034026465028
Precision:  0.9704257393565161
Recall:  0.9825600526488977
F1:  0.975784845050756

Confusion Matrix:
 [[11285   364]
 [  212 11944]]

Top 5 Negative Features:
         Feature  Importance
54379    reuters  -13.585771
1202   breitbart   -6.155239
55638       said   -5.761601
63968     trumps   -4.003140
9442         000   -3.975436

Top 5 Positive Features:
        Feature  Importance
66079       via    6.541377
36281     image    3.928148
9014      video    3.668262
46844   october    3.488561
1196   breaking    3.053872


### Experiment 5 (Hinge Loss)

In [12]:
results.append(
    run_experiment(
        df_raw,
        model= LinearSVC(random_state=42, max_iter=5000, loss='hinge'),
        vectorizer="tfidf",
        experiment_name="Hinge Loss",
        ngramRange=(1,2)
    )
)


Features created in 43.5353 seconds.
Total features =  477076
Training completed in 5.7790 seconds.
Accuracy:  0.9789540012602395
Precision:  0.974281761211036
Recall:  0.9847811780190853
F1:  0.9789388730834505

Confusion Matrix:
 [[11333   316]
 [  185 11971]]

Top 5 Negative Features:
                   Feature  Importance
352973             reuters  -13.472465
361547                said   -6.975364
1810             breitbart   -6.755870
432972              trumps   -4.495207
454520  washington reuters   -3.914274

Top 5 Positive Features:
          Feature  Importance
446457        via    6.044801
15125       video    3.929499
205873      image    3.877243
205953  image via    3.363288
288025    october    3.082336


## Model 2

Model - Linear SVM     
Vectorization - CountVectorizer         
Max iterations - 20000

### Baseline:
Features -      

* CountVectorizer  
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time :  42.2s 
* Training Time :  97.4s          

Metrics -           
* Accuracy:  0.97211
* Precision:  0.96753
* Recall:  0.97820
* F1:  0.97208
* Confusion Matrix:         
    [11250   399        -> False Positives                   
     265     11891]     -> False Negatives              

In [13]:
model = LinearSVC(random_state=42, max_iter=20000)

In [14]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="Baseline",
        ngramRange=(1,2)
    )
)


Features created in 42.7429 seconds.
Total features =  477076
Training completed in 108.9972 seconds.
Accuracy:  0.9721067002730519
Precision:  0.9675345809601302
Recall:  0.9782000658111221
F1:  0.9720864610149446

Confusion Matrix:
 [[11250   399]
 [  265 11891]]

Top 5 Negative Features:
          Feature  Importance
352973    reuters   -1.045856
1810    breitbart   -1.040491
11602     roundup   -0.433599
5889      graphic   -0.432215
15986         000   -0.375785

Top 5 Positive Features:
             Feature  Importance
15125          video    0.511844
446457           via    0.511139
1777        breaking    0.326165
143497  email donald    0.317425
281254    news trump    0.306987


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## Experiments

### Experiment 1 (Varying regularization parameter C)

In [15]:
C_var = [0.01, 0.1, 0.5, 2, 10] # Default = 1.0
for var in C_var:
    results.append(
    run_experiment(
        df_raw,
        model=LinearSVC(random_state=42,max_iter=20000,C=var),
        vectorizer="count",
        experiment_name=f"C={var}",
        ngramRange=(1,2)
    )
)


Features created in 45.5221 seconds.
Total features =  477076
Training completed in 8.5110 seconds.
Accuracy:  0.9760554505356017
Precision:  0.9698296836982968
Recall:  0.9837117472852912
F1:  0.9760358386167396

Confusion Matrix:
 [[11277   372]
 [  198 11958]]

Top 5 Negative Features:
                   Feature  Importance
352973             reuters   -0.884421
1810             breitbart   -0.826315
15986                  000   -0.319395
454520  washington reuters   -0.292691
15915           york times   -0.289864

Top 5 Positive Features:
             Feature  Importance
15125          video    0.420668
446457           via    0.391164
1777        breaking    0.257274
424665  told reuters    0.193604
288025       october    0.163966

Features created in 43.7730 seconds.
Total features =  477076
Training completed in 44.0435 seconds.
Accuracy:  0.9739130434782609
Precision:  0.9689405642735182
Recall:  0.9803389272787101
F1:  0.9738935792282115

Confusion Matrix:
 [[11267   382]
 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Training completed in 110.3044 seconds.
Accuracy:  0.9719806763285024
Precision:  0.9674505655464236
Recall:  0.978035538005923
F1:  0.9719604090586544

Confusion Matrix:
 [[11249   400]
 [  267 11889]]

Top 5 Negative Features:
          Feature  Importance
352973    reuters   -1.053442
1810    breitbart   -1.047519
11602     roundup   -0.469518
5889      graphic   -0.466765
13753       times   -0.384215

Top 5 Positive Features:
             Feature  Importance
446457           via    0.515743
15125          video    0.514229
143497  email donald    0.344609
281254    news trump    0.332955
1777        breaking    0.327675

Features created in 46.7733 seconds.
Total features =  477076
Training completed in 99.9875 seconds.
Accuracy:  0.9715605965133375
Precision:  0.967271839127249
Recall:  0.9773774267851267
F1:  0.9715404095594935

Confusion Matrix:
 [[11247   402]
 [  275 11881]]

Top 5 Negative Features:
          Feature  Importance
352973    reuters   -1.067752
1810    breitbar

### Experiment 2 (Varying min_df)

In [16]:
mdf_var = [2, 10, 20] 
for var in mdf_var:
    results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name=f"mdf={var}",
        ngramRange=(1,2),
        mdf=var
    )
)


Features created in 46.5661 seconds.
Total features =  550000
Training completed in 106.0935 seconds.
Accuracy:  0.9724427641251838
Precision:  0.9680123718053069
Recall:  0.9783645936163211
F1:  0.9724230176659052

Confusion Matrix:
 [[11256   393]
 [  263 11893]]

Top 5 Negative Features:
                   Feature  Importance
412950             reuters   -1.040213
3627             breitbart   -1.027944
30530              roundup   -0.435991
14900              graphic   -0.434406
523666  washington reuters   -0.366275

Top 5 Positive Features:
             Feature  Importance
515119           via    0.501664
45708          video    0.487232
3558        breaking    0.320704
186232  email donald    0.316441
336974    news trump    0.306571

Features created in 42.4162 seconds.
Total features =  195987
Training completed in 68.3252 seconds.
Accuracy:  0.9707624448645242
Precision:  0.9668404758025094
Recall:  0.9762257321487331
F1:  0.9707422768014969

Confusion Matrix:
 [[11242   407]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


### Experiment 3 (Unigrams Only)

In [17]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="Unigrams Only",
        ngramRange=(1,1)
    )
)


Features created in 15.0472 seconds.
Total features =  70042
Training completed in 58.9042 seconds.
Accuracy:  0.9665616467128755
Precision:  0.9623901009443178
Recall:  0.9725238565317539
F1:  0.9665378364002777

Confusion Matrix:
 [[11187   462]
 [  334 11822]]

Top 5 Negative Features:
         Feature  Importance
1202   breitbart   -1.378563
54379    reuters   -1.300456
8530       times   -0.696062
3968        held   -0.677154
9442         000   -0.643142

Top 5 Positive Features:
            Feature  Importance
66079           via    0.815636
9014          video    0.699232
1196       breaking    0.642989
62619       thomson    0.416651
54380  reutersipsos    0.382407


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


### Experiment 4 (Hinge Loss)

In [18]:
results.append(
    run_experiment(
        df_raw,
        model= LinearSVC(random_state=42, max_iter=20000, loss='hinge'),
        vectorizer="count",
        experiment_name="Hinge Loss",
        ngramRange=(1,2)
    )
)


Features created in 46.0600 seconds.
Total features =  477076
Training completed in 12.9907 seconds.
Accuracy:  0.971686620457887
Precision:  0.9672798307016116
Recall:  0.9776242184929254
F1:  0.9716663321750307

Confusion Matrix:
 [[11247   402]
 [  272 11884]]

Top 5 Negative Features:
          Feature  Importance
352973    reuters   -1.077133
1810    breitbart   -1.064914
11602     roundup   -0.522891
5889      graphic   -0.517611
13753       times   -0.403825

Top 5 Positive Features:
             Feature  Importance
446457           via    0.525109
15125          video    0.513582
143497  email donald    0.374468
281254    news trump    0.362629
1777        breaking    0.331543


### Maximized parameters ver
* TF-IDF
* Hinge Loss
* C = 10
* min_df = 2

In [ ]:
results.append(
    run_experiment(
        df_raw,
        model= LinearSVC(random_state=42, max_iter=5000, loss='hinge', C=10),
        vectorizer="tfidf",
        experiment_name="Hinge Loss+C=10+mdf=2",
        mdf=2,
        ngramRange=(1,2)
    )
)


Features created in 46.5163 seconds.
Total features =  550000
Training completed in 29.8856 seconds.
Accuracy:  0.9795001050199538
Precision:  0.9760117493472585
Recall:  0.9840408028956894
F1:  0.9794867674990542

Confusion Matrix:
 [[11355   294]
 [  194 11962]]

Top 5 Negative Features:
                   Feature  Importance
412950             reuters  -13.195954
3627             breitbart   -7.081320
422179                said   -6.777492
500749              trumps   -4.290277
523666  washington reuters   -3.902631

Top 5 Positive Features:
          Feature  Importance
515119        via    5.995977
45708       video    4.019658
255233      image    3.748282
255316  image via    3.325173
3558     breaking    3.131397


In [20]:
comparison_df = pd.DataFrame(results)
comparison_df.sort_values("Accuracy", ascending=False)

,Experiment,Vectorizer,Accuracy,Precision,Recall,F1,Features,Feature Time (s),Training Time (s)
23,Hinge Loss+C=10+mdf=2,tfidf,0.979500,0.976012,0.984041,0.979487,550000,46.516332,29.885627
6,mdf=2,tfidf,0.979332,0.974455,0.985357,0.979317,550000,44.214766,0.942669
7,mdf=10,tfidf,0.979122,0.973751,0.985686,0.979106,195987,43.746193,0.707805
11,Hinge Loss,tfidf,0.978954,0.974282,0.984781,0.978939,477076,43.535263,5.779044
5,C=10,tfidf,0.978870,0.974123,0.984781,0.978855,477076,43.346496,4.262267
4,C=2,tfidf,0.978618,0.973725,0.984699,0.978602,477076,44.712655,1.318211
0,Baseline,tfidf,0.978408,0.972944,0.985110,0.978391,477076,43.643485,0.912600
3,C=0.5,tfidf,0.978282,0.972324,0.985522,0.978265,477076,43.987842,0.676833
8,mdf=20,tfidf,0.978198,0.972549,0.985110,0.978181,87429,44.034945,0.613363
13,C=0.01,count,0.976055,0.969830,0.983712,0.976036,477076,45.522100,8.510962


In [21]:
comparison_df.to_csv(
    "../results/linear_svm.csv",
    index=False
)

## Model Analysis and Experimental Findings

### Baseline Performance

Linear SVM with TF-IDF vectorization achieved:

- Accuracy: 97.84% 
- Precision: 97.29% 
- Recall: 98.51% 
- F1 Score: 97.84% 

The baseline Linear SVM outperformed Logistic Regression across all evaluation metrics, although the improvement was relatively small.

---

## Key Findings

* Linear SVM achieved the best overall performance among all evaluated models.

* The best configuration used TF-IDF features with bigrams, `min_df=2`, hinge loss, and `C=10`, achieving:

  * Accuracy: **97.95%**
  * Precision: **97.60%**
  * Recall: **98.40%**
  * F1 Score: **97.95%**

* TF-IDF consistently outperformed Count Vectorizer for Linear SVM.

* Count Vectorizer required substantially longer training times while producing lower accuracy.

* Increasing the regularization parameter (`C`) above the default value provided only marginal performance improvements at the cost of longer training times.

* Lower values of `min_df` slightly improved performance by retaining more rare terms, although this substantially increased feature dimensionality.

* Removing source identifiers reduced performance by approximately 0.7 percentage points, indicating that publisher-related information contributes useful signals but is not solely responsible for the model's success.

* Unigram-only models performed worse than unigram-plus-bigram models, demonstrating that phrase-level information improves classification.

* Linear SVM was more robust to preprocessing changes than Naive Bayes and consistently outperformed both Naive Bayes and Logistic Regression.

* The strong performance of both Logistic Regression and Linear SVM suggests that the dataset is close to linearly separable in TF-IDF feature space.